In [1]:
!rm -rf .five_minute_cache

# Fleche in Five Minutes

*A persistent cache for expensive Python functions — `lru_cache` on steroids.*

Decorate a function and `fleche` stores every result under a SHA256 key built from the
function's identity and the **content** of its arguments. Results survive restarts, can
live in files, HDF5, SQL, or on another machine — and everything you ever computed stays
queryable like a small database.

If your day involves functions that take minutes to days — structure relaxations,
phonons, MD, training runs — and you re-run them more often than you'd like, this is
for you.

```
pip install fleche          # or: conda install -c conda-forge fleche
```

## 1. Decorate and forget

Point the active cache at a directory (in real projects you'd do this once in a
`fleche.toml` next to your code — see the end of this notebook), then just decorate:

In [2]:
import time
from fleche import fleche, cache, tags, wrap_executor
from fleche.caches import Cache

cache(Cache.from_config({"template": "cloudpickle", "root": "./.five_minute_cache"}));

In [3]:
@fleche
def expensive(x):
    print(f"crunching {x} ...")
    time.sleep(2)                     # pretend this is a real calculation
    return x ** 2


start = time.time()
print(expensive(4), f"  <- first call: {time.time() - start:.2f} s")

start = time.time()
print(expensive(4), f"  <- second call: {time.time() - start:.4f} s")

crunching 4 ...


16   <- first call: 2.00 s
16   <- second call: 0.0016 s


The second call never enters the function body. Unlike `lru_cache`, the result is on
disk, not in process memory — restart the kernel (skip the cleanup cell at the top) and
it is still instant. A brand-new cache object pointed at the same directory already
knows the answer:

In [4]:
fresh = Cache.from_config({"template": "cloudpickle", "root": "./.five_minute_cache"})
with cache(fresh):
    print("cached?", expensive.fleche.contains(4))

cached? True


The cache is just a directory — `rsync` it, back it up, share it with your group.

## 2. Keys are content, not object identity

Arguments are digested by **value** — not `id()`, not pickle bytes. NumPy arrays,
pandas frames, dataclasses, and nested containers work out of the box; equal content
means equal key, no matter who constructed the object:

In [5]:
import numpy as np


@fleche
def norm(arr):
    print("computing ...")
    return float(np.linalg.norm(arr))


a = np.linspace(0, 1, 1_000_000)
print(norm(a))
print(norm(a.copy()))                 # different object, same content -> cache hit

computing ...
577.3504135273193
577.3504135273193


The digest machinery is pluggable for third-party types. If you use ASE:
`pip install fleche-ase` and `Atoms`, `Calculator`, and `VibrationsData` objects digest
correctly with **zero further setup** (registered via entry points):

In [6]:
try:
    from ase.build import bulk
    from ase.calculators.emt import EMT

    @fleche
    def energy(atoms):
        print("running EMT ...")
        atoms = atoms.copy()
        atoms.calc = EMT()
        return atoms.get_potential_energy()

    print(energy(bulk("Cu", cubic=True)))
    print(energy(bulk("Cu", cubic=True)))   # freshly built Atoms -> cache hit
except ImportError:
    print("pip install ase fleche-ase to run this cell")

running EMT ...
-0.022726045434316333
-0.022726045434316333


## 3. Your cache is a database

Every call is recorded — arguments, runtime, metadata — *separately* from the (possibly
heavy) result values, so you can browse what you computed without deserializing any of
it. Tag calls, then query into a pandas DataFrame:

In [7]:
@fleche
def relax(a, k):
    time.sleep(0.1)                   # pretend
    return {"energy": -a * k, "volume": a ** 3}


with tags(project="five-minute-demo"):
    for a_lat in (3.5, 3.6, 3.7):
        relax(a_lat, k=4)

relax.fleche.query().table(arguments=["a", "k"], results=True)

,name,module,result,timestart,timestop,walltime,project,a,k
ad2b,relax,__main__,"{'energy': -14.0, 'volume': 42.875}",2026-08-07 17:27:11.237918139+00:00,2026-08-07 17:27:11.339107752+00:00,0.101190,five-minute-demo,3.5,4
0275,relax,__main__,"{'energy': -14.4, 'volume': 46.656000000000006}",2026-08-07 17:27:11.340700626+00:00,2026-08-07 17:27:11.441451073+00:00,0.100750,five-minute-demo,3.6,4
ba58,relax,__main__,"{'energy': -14.8, 'volume': 50.653000000000006}",2026-08-07 17:27:11.443126440+00:00,2026-08-07 17:27:11.543898106+00:00,0.100772,five-minute-demo,3.7,4


Queries chain (`filter`, `unique`, `sorted`, ...) and terminal methods can `transfer()`
matching entries to another cache or `evict()` them; `cache().table()` shows everything
across all functions.

## 4. Plays well with executors — including executorlib

`wrap_executor` patches any `concurrent.futures`-style executor. Fleche-decorated
functions carry the active cache into worker processes automatically, and cache hits
come back as already-completed futures **without ever being submitted**:

In [8]:
try:
    from executorlib import SingleNodeExecutor as Executor
except ImportError:
    from concurrent.futures import ProcessPoolExecutor as Executor


@fleche
def md_step(x):
    time.sleep(1)                     # pretend
    return x ** 3


for attempt in ("cold", "warm"):
    start = time.time()
    with Executor(max_workers=4) as ex:
        wrap_executor(ex)
        results = [f.result() for f in [ex.submit(md_step, x) for x in range(4)]]
    print(f"{attempt}: {results} in {time.time() - start:.2f} s")

cold: [0, 1, 8, 27] in 2.34 s
warm: [0, 1, 8, 27] in 0.03 s


On the warm pass every future is done before `submit` returns — the executor never sees
the work. Executor-specific kwargs like executorlib's `resource_dict` are forwarded
transparently.

## 5. Stack caches — a read-only, prepopulated base

Say your group already has a cache full of results — on a shared filesystem, or a
teammate's directory. `CacheStack` puts your own writable cache *in front* of it: loads
fall through to the base and hits are back-filled into the front, while saves only ever
touch the front. Wrap the base in `ReadOnlyCache` and nothing you do can modify it —
writes and evictions raise `Rejected`:

In [9]:
from fleche.caches import CacheStack, ReadOnlyCache
from fleche.storage.memory import ValueMemory, CallMemory


@fleche
def phonon_dos(structure):
    print(f"computing DOS for {structure} ...")
    time.sleep(1)                     # pretend
    return f"dos({structure})"


# the group's prepopulated cache (in memory here; files or SSH in real life)
shared = Cache(ValueMemory({}), CallMemory({}))
with cache(shared):
    phonon_dos("fcc-Al")

mine = Cache(ValueMemory({}), CallMemory({}))
stack = CacheStack((mine, ReadOnlyCache(shared)))

with cache(stack):
    start = time.time()
    print(phonon_dos("fcc-Al"), f"  <- from the shared base: {time.time() - start:.3f} s")
    print(phonon_dos("bcc-Fe"), "  <- not in the base: computed here")

computing DOS for fcc-Al ...


dos(fcc-Al)   <- from the shared base: 0.001 s
computing DOS for bcc-Fe ...


dos(bcc-Fe)   <- not in the base: computed here


In [10]:
with cache(mine):
    print("back-filled into my cache:", phonon_dos.fleche.contains("fcc-Al"))
    print("new result in my cache:   ", phonon_dos.fleche.contains("bcc-Fe"))
with cache(shared):
    print("shared base untouched:    ", not phonon_dos.fleche.contains("bcc-Fe"))

back-filled into my cache: True
new result in my cache:    True
shared base untouched:     True


The same stack straight from `fleche.toml` — an array of tables, first entry is the
front where saves land:

```toml
[[stacked]]
template = "memory"

[[stacked]]
template = "cloudpickle"
root = "/groupshare/fleche-cache"
read_only = true
```

## 6. Things worth a second look

- **Configuration by file** — drop a `fleche.toml` next to your project (or in `$HOME`);
  decorated code never changes:

  ```toml
  [default]
  cache = "persistent"

  [persistent]
  template = "cloudpickle"
  root = "~/.cache/fleche"
  ```

- **HDF5 storage** — `template = "bagofholding_hdf"` stores values via pyiron's
  [bagofholding](https://github.com/pyiron/bagofholding).
- **SQL call index** — keep call records in SQLite/Postgres for server-side filtering
  over large caches, while values stay in files or HDF5.
- **Caches on other machines** — `SshCache` forwards a whole cache over SSH to a
  `python -m fleche remote --serve` process on, say, your cluster's login node. Stack it
  behind a local cache exactly as above, or fan reads over several teammates' caches at
  once with a read-only `CachePool`.
- **Take results home** — `query().transfer(other_cache)` moves selected results
  between caches (cluster → laptop); see `TransferWorkflow.ipynb`.
- **Invalidation you control** — bump `@fleche(version=2)` to retire old entries, or
  `hash_code=True` to invalidate on any code edit; `ignore=`/`require=` shape the key.
- **Signed storage** — HMAC-sign pickle-family entries with a `secret_key`; tampered or
  wrong-key entries surface as plain cache misses, never as executed code.

## Where to go next

- Docs: <https://fleche.readthedocs.io>
- Source: <https://github.com/pmrv/fleche>
- Deeper notebooks in this directory: `GettingStarted`, `ExtraMethods`,
  `StorageBackends`, `ConcurrentExecution`, `CacheStack`, `SecureStorage`,
  `TransferWorkflow`.